In [3]:
from ingest import load_faq_data
documents = load_faq_data()

In [4]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [5]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [6]:
documents = documents_llm

In [7]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [8]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [9]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [11]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [12]:
import json
user_prompt = json.dumps(doc)

In [13]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [14]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [15]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [16]:
result = response.output_parsed

print(result)

questions=['Is it too late to start this course if I only found it now?', 'Can I still join the course after it already started?', 'If I join late, can I still get a certificate somehow?', "What do I need to do to be eligible for the certificate if I'm coming in late?", 'Are late submissions for the project accepted, or do I need to finish it while submissions are still open?']


In [18]:
for question in result.questions:
    print(question)

Is it too late to start this course if I only found it now?
Can I still join the course after it already started?
If I join late, can I still get a certificate somehow?
What do I need to do to be eligible for the certificate if I'm coming in late?
Are late submissions for the project accepted, or do I need to finish it while submissions are still open?


In [19]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [20]:
from evaluation_utils import llm_structured

In [21]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Can I still join the course if I just found out about it?', 'Is it too late to enroll in the course now?', 'If I join late, can I still get a certificate somehow?', 'What do I need to do to be eligible for the certificate after joining late?', 'Are project submissions still open for new students who want a certificate?']


In [22]:
usage.input_tokens, usage.output_tokens

(207, 83)

In [23]:
from evaluation_utils import calc_price

In [24]:
calc_price(usage)

{'input_cost': 0.00015525,
 'output_cost': 0.00037349999999999997,
 'total_cost': 0.0005287499999999999}

In [25]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I still join the course if I just found out about it?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to enroll in the course now?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, can I still get a certificate somehow?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to be eligible for the certificate after joining late?',
  'document': '74eb249bbf'},
 {'question': 'Are project submissions still open for new students who want a certificate?',
  'document': '74eb249bbf'}]

In [26]:
import pandas as pd

In [27]:
pd.DataFrame(records)

,question,document
0,Can I still join the course if I just found ou...,74eb249bbf
1,Is it too late to enroll in the course now?,74eb249bbf
2,"If I join late, can I still get a certificate ...",74eb249bbf
3,What do I need to do to be eligible for the ce...,74eb249bbf
4,Are project submissions still open for new stu...,74eb249bbf


In [28]:
from evaluation_utils import llm_structured_retry

In [29]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [18]:
generate_ground_truth(doc)

([{'question': 'How are the capstone homework points calculated?',
   'document': '0d200c8c58'},
  {'question': 'What do I need to do to get full score on a homework?',
   'document': '0d200c8c58'},
  {'question': 'How many points can I earn from one homework assignment in this project?',
   'document': '0d200c8c58'},
  {'question': 'Is there a point breakdown for answers, learning items, and FAQ questions?',
   'document': '0d200c8c58'},
  {'question': 'What actions count toward the homework leaderboard score?',
   'document': '0d200c8c58'}],
 ResponseUsage(input_tokens=263, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=77, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=340))

In [30]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [31]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [32]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

In [33]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

565

In [34]:
ground_truth[10]

{'question': 'How do students join the Office Hours or live workshop if the Zoom link isn’t shared publicly?',
 'document': '489dd1c9d9'}

In [35]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08711700000000001

In [36]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.08711700000000001

In [37]:
df_ground_truth = pd.DataFrame(ground_truth)

In [38]:
df_ground_truth.head()

,question,document
0,Can I join the course late if I just found it ...,74eb249bbf
1,Is it too late to enroll in the course after i...,74eb249bbf
2,"If I join now, can I still get a certificate?",74eb249bbf
3,What do I need to do to receive a certificate ...,74eb249bbf
4,Are project submissions still open for new stu...,74eb249bbf


In [40]:
df_ground_truth.to_csv("../data/ground_truth.csv", index=False)

In [41]:
len(df_ground_truth)

565